# log-samples-eval-callback — ex2: eval callback with start_step warm-up offset and sink capacity cap

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `log-samples-eval-callback`. Running the final beacon cell reports progress against the `Logging: log-samples eval callback` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Logging: log-samples eval callback` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`log-samples-eval-callback`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "log-samples-eval-callback"
DD_SUBTOPIC = "Logging: log-samples eval callback"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Eval callback with start_step offset + capacity cap

Ex1 fired at `step % K == 0`. Two real-world extensions:

1. **`start_step` offset** — skip eval during the warm-up phase. Fire only when `step >= start_step AND (step - start_step) % K == 0`. Lets you skip the noisy first 1000 steps without changing K.
2. **Capacity cap** — the sink has a `max_entries`. Once full, no more appends (downstream storage limit, wandb table cap, etc.).

```python
for step in range(n_steps):
    if step >= start and (step - start) % K == 0 and len(sink) < cap:
        sink.append({'step': step, 'samples': sample()})
```

**Why aligned to `start_step`, not absolute step.** A warm-up of 100 steps with K=50 should fire at 100, 150, 200, ... — not at 100, 150, 200 only because they happen to be K-multiples. The offset shifts the cadence origin.

### Exercise 2 — eval callback with start_step warm-up offset and sink capacity cap

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply an offset-aligned modulo-K cadence with a `max_entries` guard so an eval callback skips a warm-up window AND stops appending once the sink hits capacity.
> Keywords: eval, callback, warmup, capacity
> ```

**KCs targeted:** `offset-aligned-cadence`, `capacity-bounded-sink`

Implement `ex2_run_with_offset_callback(n_steps, eval_every, start_step, n_eval, sink, max_entries)`. The deepening variant of ex1's callback.

Behaviour:

1. Loop `for step in range(n_steps)`.
2. Fire iff:
   - `step >= start_step` (warm-up gate), AND
   - `(step - start_step) % eval_every == 0` (offset-aligned cadence), AND
   - `len(sink) < max_entries` (capacity guard).
3. When firing, append `{'step': step, 'samples': [f'step={step}-sample={i}' for i in range(n_eval)]}` to `sink`.
4. Return the number of `sink.append` calls performed by THIS function (so a sink that was non-empty going in still counts only the fires this call made).

Edge cases:
- `start_step >= n_steps` → no fires.
- `max_entries == 0` → no fires regardless of cadence.
- A `sink` that's already at or above `max_entries` → no fires.

In [ ]:
def ex2_run_with_offset_callback(n_steps, eval_every, start_step, n_eval, sink, max_entries):
    n_fires = 0
    for step in range(n_steps):
        if step < start_step:
            continue
        if (step - start_step) % eval_every != 0:
            continue
        if len(sink) >= max_entries:
            break  # sink full, no further fires will succeed either
        samples = [f'step={step}-sample={i}' for i in range(n_eval)]
        sink.append({'step': step, 'samples': samples})
        n_fires += 1
    return n_fires


<details><summary>Solution</summary>

```python
def ex2_run_with_offset_callback(n_steps, eval_every, start_step, n_eval, sink, max_entries):
    n_fires = 0
    for step in range(n_steps):
        if step < start_step:
            continue
        if (step - start_step) % eval_every != 0:
            continue
        if len(sink) >= max_entries:
            break  # sink full, no further fires will succeed either
        samples = [f'step={step}-sample={i}' for i in range(n_eval)]
        sink.append({'step': step, 'samples': samples})
        n_fires += 1
    return n_fires
```

**Why `break` instead of `continue` on cap-hit.** Once `len(sink) >= max_entries`, no later step will pass the guard either — the sink only grows. `break` is correct AND faster than walking the remaining n_steps doing nothing.

**Offset alignment, not absolute alignment.** Ex1's `step % K == 0` fires at 0, K, 2K. With `start_step=5, K=3`, the fires are 5, 8, 11, 14 — aligned to the offset origin, NOT to the absolute clock. This is what users want when they say 'eval every K steps after warm-up'.

**Don't double-count.** The return value is fires from THIS call. A pre-populated sink stays unchanged in the counter. Real ARENA callbacks track per-call fires so a downstream throttle can back-off.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()